# 🧪 MolGuard — Drug-Food Interaction Model Training
### D-MPNN (Chemprop v2) on SMILES Molecular Graphs

> **First**: Runtime → Change runtime type → **T4 GPU** ✅

In [ ]:
# CELL 1 — Check GPU
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# CELL 2 — Install packages
!pip install -q chemprop==2.0.4 rdkit pandas scikit-learn matplotlib seaborn
print('All packages installed')

In [ ]:
# CELL 3 — Upload dataset
from google.colab import files
print('Upload drug_interactions.csv')
uploaded = files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
# CELL 4 — Load and validate
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

df = pd.read_csv('drug_interactions.csv')
print(f'Shape: {df.shape}')
print(f'Labels:\n{df["label"].value_counts()}')

def valid_smiles(s):
    try:
        return Chem.MolFromSmiles(s) is not None
    except:
        return False

df['drug_ok']  = df['drug_smiles'].apply(valid_smiles)
df['const_ok'] = df['constituent_smiles'].apply(valid_smiles)
df = df[df['drug_ok'] & df['const_ok']].copy()
print(f'Valid rows: {len(df)}')
df.head(3)

In [ ]:
# CELL 5 — Train/Val/Test split
from sklearn.model_selection import train_test_split
import os

cp = df[['drug_smiles','constituent_smiles','label']].rename(
    columns={'drug_smiles':'smiles_drug','constituent_smiles':'smiles_constituent'})
cp['label'] = cp['label'].astype(int)

train_df, temp = train_test_split(cp, test_size=0.30, random_state=42, stratify=cp['label'])
val_df,  test_df = train_test_split(temp, test_size=0.50, random_state=42, stratify=temp['label'])

os.makedirs('chemprop_data', exist_ok=True)
train_df.to_csv('chemprop_data/train.csv', index=False)
val_df.to_csv('chemprop_data/val.csv',   index=False)
test_df.to_csv('chemprop_data/test.csv',  index=False)
print(f'Train:{len(train_df)} Val:{len(val_df)} Test:{len(test_df)}')

In [ ]:
# CELL 6 — Build data loaders
import torch
from chemprop import data, featurizers, models
from chemprop import nn as cnn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE.upper()}')

def make_dataset(csv_path):
    df = pd.read_csv(csv_path)
    pts = []
    for _, r in df.iterrows():
        if Chem.MolFromSmiles(r.smiles_drug) and Chem.MolFromSmiles(r.smiles_constituent):
            pts.append(data.MoleculeDatapoint.from_smi(
                smis=[r.smiles_drug, r.smiles_constituent],
                y=np.array([[r.label]], dtype=float)))
    return pts

feat      = featurizers.SimpleMoleculeMolGraphFeaturizer()
train_ds  = data.MoleculeDataset(make_dataset('chemprop_data/train.csv'), featurizer=feat)
val_ds    = data.MoleculeDataset(make_dataset('chemprop_data/val.csv'),   featurizer=feat)
test_ds   = data.MoleculeDataset(make_dataset('chemprop_data/test.csv'),  featurizer=feat)

train_loader = data.build_dataloader(train_ds, shuffle=True,  batch_size=32)
val_loader   = data.build_dataloader(val_ds,   shuffle=False, batch_size=32)
test_loader  = data.build_dataloader(test_ds,  shuffle=False, batch_size=32)
print(f'Datasets ready')

In [ ]:
# CELL 7 — Build D-MPNN model
mp  = cnn.BondMessagePassing()
agg = cnn.MeanAggregation()
ffn = cnn.BinaryClassificationFFN(
    input_dim=mp.output_dim * 2,
    hidden_dim=300, n_layers=3, dropout=0.2)

model = models.MPNN(message_passing=mp, agg=agg, predictor=ffn).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total:,}')
print(model)

In [ ]:
# CELL 8 — Training loop with early stopping
import torch.nn as tnn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, accuracy_score

EPOCHS   = 50
LR       = 1e-4
PATIENCE = 10

opt     = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
sched   = ReduceLROnPlateau(opt, mode='max', patience=5, factor=0.5)
loss_fn = tnn.BCEWithLogitsLoss()

history = {'train_loss':[], 'val_loss':[], 'val_auc':[], 'val_acc':[]}
best_auc, best_epoch, patience_ctr = 0.0, 0, 0

print(f'Training for up to {EPOCHS} epochs on {DEVICE.upper()}...')

for epoch in range(1, EPOCHS+1):
    # --- Train ---
    model.train()
    train_losses = []
    for batch in train_loader:
        batch = batch.to(DEVICE)
        opt.zero_grad()
        logits  = model(batch).squeeze(-1)
        targets = batch.y.float().squeeze(-1)
        loss    = loss_fn(logits, targets)
        loss.backward()
        tnn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        train_losses.append(loss.item())

    # --- Validate ---
    model.eval()
    val_losses, val_preds, val_targets = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            logits  = model(batch).squeeze(-1)
            targets = batch.y.float().squeeze(-1)
            val_losses.append(loss_fn(logits, targets).item())
            val_preds.extend(torch.sigmoid(logits).cpu().numpy())
            val_targets.extend(targets.cpu().numpy())

    vauc = roc_auc_score(val_targets, val_preds)
    vacc = accuracy_score(val_targets, [1 if p>=0.5 else 0 for p in val_preds])
    sched.step(vauc)

    history['train_loss'].append(np.mean(train_losses))
    history['val_loss'].append(np.mean(val_losses))
    history['val_auc'].append(vauc)
    history['val_acc'].append(vacc)

    if vauc > best_auc:
        best_auc, best_epoch, patience_ctr = vauc, epoch, 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_auc': vauc}, 'best_model.pt')
        print(f'  Epoch {epoch:3d} | NEW BEST AUC={vauc:.4f} Acc={vacc:.4f} | saved')
    else:
        patience_ctr += 1
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | AUC={vauc:.4f} Acc={vacc:.4f} | patience {patience_ctr}/{PATIENCE}')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'Best epoch: {best_epoch} | Best Val AUC: {best_auc:.4f}')

In [ ]:
# CELL 9 — Plot training curves
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({'text.color':'#e8f4f0','axes.labelcolor':'#7a9e95',
                     'xtick.color':'#7a9e95','ytick.color':'#7a9e95'})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#050a0f')
for ax in axes:
    ax.set_facecolor('#0a1520')
    for spine in ax.spines.values():
        spine.set_color('#1a3a32')

ep = range(1, len(history['train_loss'])+1)
axes[0].plot(ep, history['train_loss'], color='#ff4757', label='Train')
axes[0].plot(ep, history['val_loss'],   color='#00c8aa', label='Val')
axes[0].set_title('Loss Curves'); axes[0].legend()

axes[1].plot(ep, history['val_auc'], color='#00c8aa')
axes[1].axhline(best_auc, color='#ff4757', ls='--', alpha=0.7, label=f'Best {best_auc:.3f}')
axes[1].set_title('Validation AUC-ROC'); axes[1].legend()

axes[2].plot(ep, history['val_acc'], color='#ffa502')
axes[2].set_title('Validation Accuracy')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#050a0f')
plt.show()
print('Saved training_curves.png')

In [ ]:
# CELL 10 — Test set evaluation + confusion matrix
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import seaborn as sns

ckpt = torch.load('best_model.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

test_preds, test_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(DEVICE)
        test_preds.extend(torch.sigmoid(model(batch).squeeze(-1)).cpu().numpy())
        test_targets.extend(batch.y.float().squeeze(-1).cpu().numpy())

test_bin = [1 if p>=0.5 else 0 for p in test_preds]
test_auc = roc_auc_score(test_targets, test_preds)
test_acc = accuracy_score(test_targets, test_bin)

print('='*50)
print(f'Test AUC-ROC : {test_auc:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print('='*50)
print(classification_report(test_targets, test_bin, target_names=['Safe','Unsafe']))

fig, ax = plt.subplots(figsize=(5,4))
fig.patch.set_facecolor('#050a0f'); ax.set_facecolor('#0a1520')
sns.heatmap(confusion_matrix(test_targets, test_bin), annot=True, fmt='d',
            cmap='YlOrRd', xticklabels=['Safe','Unsafe'], yticklabels=['Safe','Unsafe'], ax=ax)
ax.set_title('Confusion Matrix', color='#e8f4f0')
plt.savefig('confusion_matrix.png', dpi=150, facecolor='#050a0f')
plt.show()

In [ ]:
# CELL 11 — Save model artifacts for backend
import json, shutil

os.makedirs('model_artifacts', exist_ok=True)

config = {
    'model_type':    'D-MPNN-BinaryClassifier',
    'n_molecules':   2,
    'hidden_dim':    300,
    'n_layers':      3,
    'dropout':       0.2,
    'threshold':     0.5,
    'best_epoch':    int(best_epoch),
    'best_val_auc':  round(float(best_auc), 4),
    'test_auc':      round(float(test_auc), 4),
    'test_accuracy': round(float(test_acc), 4),
}

with open('model_artifacts/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

torch.save({'model_state_dict': model.state_dict(), 'model_config': config},
           'model_artifacts/full_checkpoint.pt')

shutil.copy('training_curves.png',  'model_artifacts/')
shutil.copy('confusion_matrix.png', 'model_artifacts/')

print('Artifacts saved:')
print(json.dumps(config, indent=2))

In [ ]:
# CELL 12 — Quick inference test (sanity check)
from chemprop import data as cdata, featurizers as cfeat

model.eval()

def quick_predict(drug_smi, const_smi, threshold=0.5):
    dp = cdata.MoleculeDatapoint.from_smi(
        smis=[drug_smi, const_smi], y=np.array([[0.]]))
    ds = cdata.MoleculeDataset([dp], featurizer=cfeat.SimpleMoleculeMolGraphFeaturizer())
    loader = cdata.build_dataloader(ds, shuffle=False, batch_size=1)
    with torch.no_grad():
        for b in loader:
            prob = torch.sigmoid(model(b.to(DEVICE)).squeeze(-1)).item()
    label = 'UNSAFE' if prob >= threshold else 'SAFE'
    return {'probability': round(prob, 4), 'label': label}

tests = [
    ('Warfarin + Vitamin K [expect UNSAFE]',
     'CC1(C2CC3CC(C2)(CC3C1=O)OC(=O)c1ccccc1)O',
     'CC1(CCC(=C)C(C1)OC(=O)/C=C/c1ccc(O)cc1)C'),
    ('MAOIs + Tyramine [expect UNSAFE]',
     'NNCc1ccccc1',
     'NCCc1ccc(O)cc1'),
    ('Metformin + Lycopene [expect SAFE]',
     'CN(C)C(=N)NC(=N)N',
     'CC(=C/C=C/C(=C/C=C/C(=C/C=C/C(C)(C)C)C)C)C=CC=C(C)C=CC=C(C)C'),
]

for name, dsmi, csmi in tests:
    r = quick_predict(dsmi, csmi)
    icon = 'PASS' if ('UNSAFE' in name and r['label']=='UNSAFE') or ('SAFE' in name and r['label']=='SAFE') else 'FAIL'
    print(f'[{icon}] {name} -> {r["label"]} ({r["probability"]})')

In [ ]:
# CELL 13 — Download everything
from google.colab import files
import shutil

shutil.make_archive('model_artifacts', 'zip', 'model_artifacts')
files.download('model_artifacts.zip')

print()
print('Download complete!')
print('Next steps:')
print('  1. Extract model_artifacts.zip')
print('  2. Place model_artifacts/ folder inside your backend/ directory')
print('  3. Run: uvicorn main:app --reload  (in backend/)')
print('  4. Run: npm run dev  (in frontend/)')